\setcounter{secnumdepth}{0}

# Data-preparatie

Dit notebook behandelt de data-preparatie en geeft antwoord op deelvraag 1:

*"Hoe kan cellijn-data (mutaties, methylatie, drug-respons) worden verzameld, opgeschoond en geschikt gemaakt voor machine learning?"*

Hierbij is de volgende subdeelvraag tot stand gekomen: 

*"Hoe kunnen mutatie- en methylatiefeatures worden samengevat naar gen- of pathway-niveau om dimensionaliteit te reduceren en biologische interpretatie te vergemakkelijken?"*

Doel: In dit notebook wordt een overzicht gegeven van de reeds opgeschoonde datasets (methylatie, mutatie en respons) en worden deze samengevoegd voor verdere analyse. Vervolgens wordt een pathway-aggregatie uitgevoerd om de grote hoeveelheid features te reduceren en geschikt te maken voor machine learning.

___

## Achtergrond van de datasets

Voor dit project worden vier afzonderlijke datasets gebruikt die afkomstig zijn van het DepMap/CCLE-project. Deze datsets beschrijven verschillende moleculaire kenmerken en annotaties van kankercellijnen.

#### **DNA-methylatie**

Bestand (ruwe data): *`CCLE_RRBS_TSS1kb_20181022.txt`*

Deze dataset bevat DNA-methylatiegegevens gemeten met Reduced Representation Bisulfite Sequencing (RBBS). De metingen zijn uitgevoerd rond de transcription start site (TSS) van genen, binnen een bereik van ±1 kilobase (1kb). Dit gebied omvat zowel de promotorregio (upstream van de TTS) als het 5-end gedeelte van het gen, waaronder vaak het eerste exon of de 5'UTR (downstream van de TSS).

DNA-methylatie in deze regio's is biologisch relevant omdat het direct invloed kan hebben op genexpressie. Methylatie van CpG-sites kan de binding van transcriptiefactoren verhinderen en leidt vaak tot een compactere chromatinestructuur, waardoor het DNA minder toegankelijk wordt voor de transcriptiemachinerie.

- Upstream van de TSS (~1000 bp) bevindt zich de promotor. Methylatie in deze regio wordt sterk geassocieerd met onderdrukking van transcriptie.
- Downsteam van de TSS (~1000 bp) omvat vaak het 5'-uiteinde van het gen, inclusief exon 1 of de 5'UTR. Methylatie in deze regio kan de initiatie en efficientie van transcriptie beinvloeden en zo de genexpressie moduleren.

Elke rij in deze dataset representeert een CpG-locus, met de volgende belangrijke metadata:
- `CpG_sites_hg19`: genomische coördinten van de CpG-sites in het hg19-referentiegenoom.
- `avg_coverage`: de gemiddelde sequencing coverage voor dat locus, wat een maat is voor de betrouwbaarheid van de methylatiemeting.

Loci met lage coverage worden beschouwd als minder betrouwbaar en zijn daarom onderwerp van de kwaliteitscontrole.

#### **Mutaties**

Bestand (ruwe data): *`OmicsSomaticMutations.csv`*

Deze dataset bevat de somatische mutatieprofielen van kankercellijnen. Deze mutaties zijn verkregen met behulp van Whole Exome Sequencing (WES). Deze sequencingtechniek richt zich specifiek op  de coderende delen van het genoom (exons), die samen ongeveer 1-2% van het totale DNA uitmaken. Ondanks dat dit maar een klein deel van het genoom is, bevatten zij het merendeel van de mutaties die een direct effect hebben op de eiwitstructuur en eiwit functie. Mutaties in deze regio's kunnen leiden tot aminozuurveranderingen, frameshifts of voortijdige stopcodons, wat directe gevolgen kan hebben voor cellulaire processen als proliferatie, apoptose en DNA-schadeherstel. 

Niet-coderende regio's, zoals introns en regulatoire elementen, kunnen ook een functionele rol spelen in genregulatie en splicing, maar de effecten van mutaties in deze regio's zijn vaak indirect en moeilijker te voorspellen. Daarnaast leidt het meenemen van het volledige genoom tot een sterke toename in datavolume en complexiteit. In de context van dit onderzoek, waarin het aantal beschikbare cellijnen berperkt is, zou dit de kans op ruis en overfitting in machine learning-modellen sterk vergroten.


#### **Drug-respons**

Bestand (ruwe data):`*Drug_sensitivity_AUC_(PRISM_Repurposing_Secondary_Screen)_subsetted.csv*`

Deze dataset bevat responswaarden van cellijnen op verschillende geneesmiddelen, gemeten als **AUC (Area Under  the Curve) van dosis-response curves.

AUC wordt gebruikt omdat het de volledige dosis-responsrelatie samenvat over alle gemeten concentraties. Hierdoor is het minder gevoelig voor meetruis en beter gedefinieerd dan een enkelvoudige maat zoals IC50, die slechts één punt op de dosis-responscurve representeert. Daarnaast levert AUC een stabiele continue responsvariabele op, wat gunstiger is voor regressiemodellen zoals Random Forest.

Elke kolom (naast de cellijn-identifiers) representeert de respons op één geneesmiddel.


#### **Cellijn-annotaties**

Bestand: `*Cell_lines_annotations_20181226.txt*`

Dit metadatabestand bevat annotiaties voor elke cellijn, waaronder:
- `depMapID`,
- weefseltype (`Site_Primary`)
- histologie en andere aanvullende biologische kenmerken

Deze dataset wordt gebruikt om cellijnen te filteren op longkanker en om identifiers tussen de verschillende datasets te harmoniseren.


## Samenvatting preprocessing

[hier komt het samenvattende flowschema waarin de stappen te zien zijn en aantal featuers en cellijn voor en na]


De uitgebreide data-preprocessing is uitgevoerd en beschreven in de volgende notebooks:
- dataprep_methylation.ipynb
- dataprep_mutation.ipynb
- dataprep_response.ipynb


## Data-integratie/samenvoegen

Doel: Het samenvoegen van de afzonderlijke opgeschoonde datasets (mutaties, methylatie en geneesmiddelrespons) tot één geintegreerde dataset op basis van een gedeelde identifier (ModelID).

Deze stap maakt het mogelijk om:
- de overlap tussen datasets te controleren,
- inzicht te krijgen in het uiteindelijke aantal cellijnen en features,
- en een dataset te creëren die direct geschikt is voor supervised machine learning.

Eerst worden de geprepareerde datasets ingeladen. In eerdere notebooks zijn de datasets afzonderlijk voorbereid, opgeschoond en opgeslagen als pickle-bestanden. Pickle bestanden worden gebruikt om DataFrames exact met datatype en index te bewaren, zodat de data later eenvoudig kan worden ingeladen. 

Vervolgens wordt gecontroleerd of de index van elke dataset correct is ingesteld op `ModelID`. Dit voorkomt fouten bij het samenvoegen en zorgt dat de dataset op dezelfde unieke identifier worden gematcht.

Daarna wordt een inner join uitgevoerd op de ModelID-index van alle drie datasets. Dit houdt in dat alleen cellijnen die in alle drie datasets volledigeg overeenkomende rijen bevat zodat downstream analyses geen ontbrekende cellijnen bevatten. 

Tot slot wordt ter controle de vorm van de gecombineerde dataset weergegeven, zodat snel kan worden gecontroleerd hoeveel cellijnen en features beschikbaar zijn.

In [148]:
import pandas as pd
import numpy as np

# inladen van de pickles
mut_df = pd.read_pickle("../data/processed/mut_matrix.pkl")
meth_df = pd.read_pickle("../data/processed/meth_T.pkl")
resp_df = pd.read_pickle("../data/processed/response_matrix.pkl")

# zet ModelID als index en controleer
for df, name in zip([mut_df, meth_df, resp_df], ["Mutatie", "Methylatie", "Respons"]):
    if df.index.name != 'ModelID':
        if 'ModelID' in df.columns:
            df.set_index('ModelID', inplace=True)
            print(f"Waarschuwing: {name} index was niet correct. Hernoemd en kolom ModelID gebruikt als index.")
        else:
            print(f"Fout: {name} bevat geen ModelID-kolom. Controleer data!")
    else:
        print(f"{name} index correct ingesteld.")

# intersectie van cellijnen
common_cell_lines = mut_df.index.intersection(meth_df.index).intersection(resp_df.index)
print(f"\nAantal cellijnen in alle datasets: {len(common_cell_lines)}")

# gecombineerde feature-matrix
combined_df = pd.concat([
    mut_df.loc[common_cell_lines],
    meth_df.loc[common_cell_lines]
], axis=1)

# responsvector
y = resp_df.loc[common_cell_lines]

# info
print("Vorm van de gecombineerde dataset:", combined_df.shape)
print("Mutatiefeatures:", mut_df.shape[1], "Methylatiefeatures:", meth_df.shape[1], "AUC-responswaarden:", resp_df.shape[1])


Mutatie index correct ingesteld.
Methylatie index correct ingesteld.
Respons index correct ingesteld.

Aantal cellijnen in alle datasets: 82
Vorm van de gecombineerde dataset: (82, 18025)
Mutatiefeatures: 7358 Methylatiefeatures: 10667 AUC-responswaarden: 1450


Conclusie: Het aantal features in de gecombineerde dataset (mutaties + methylatie) is met 18025 veel hoger dan het aantal samples (82 cellijnen). Om te voorkomen dat dit leidt tot overfitting in het Random Forest-model, wordt in de volgende stap een pathway-aggregatie toegepast om de features biologisch te groeperen en het aantal features te reduceren.

## Pathway aggregatie


Doel: Het aantal features in de gecombineerde dataset (`combined_df`) reduceren door deze te aggregeren naar pathway-niveau. In plaats van duizenden individuele genen of CpG-loci te gebruiken, vertegenwoordigt elke pathway één samengevatte feature. Hierdoor wordt de dataset overzichtelijker en beter beheersbaar voor machine learning.

Voor aggregatie wordt gebruik gemaakt van de MSigDB (Molecular Signatures Database), een veelgebruikte en goed gedocumenteerde recourse in de bioinformatica en kankeronderzoek. MSigDB bevat verschillende verzamelingen van pathways. In dit project zijn de Hallmarks gene sets van MSigDB gebruikt [link]. Deze collectie bestaat uit 50 samengestelde pathways die belangrijke biologische processen representateren, waaronder proliferatie, DNA-schadeherstel, apoptose en oncogene signaalroutes. 

#### koppelen van genen aan pathways.


Voordat aggregatie kan plaatsvinden, moeten individuele features worden gekoppeld aan hun bijbehorende pathways.

Voor de methylatiedata betekent dit een extra stap: de oorspronkelijke featuers zijn CpG-loci met een naamgeving waarin het gen in de genomische locatie zijn opgenomen (bijvoorbeeld `AZIN2_1_33545713_33546713`). Voor aggregatie worden deze kolomnamen eerst teruggebracht tot het bijbehorende gen (zoals in het voorbeeld `AZIN2`), zodat methylatie- en mutatiegegevens op hetzelfde gen-niveau kunnen worden gekoppeld aan pathways.

Aanpak: De Hallmark gene sets worden ingelezen, genen worden gekoppeld aan pathways, en controles worden uitgevoerd om te verifieren dat alle pathways correct zijn ingelezen.

In [149]:
import pandas as pd

# kolomnamen van methylatie omzetten naar gen-symbolen 
# dit is een tussenstap voor pathway aggregatie en niet relevant voor eindverslag

# het gen uit de locusnaam halen
def extract_gene(locus_name):
    return locus_name.split('_')[0]

# kolomnamen omzetten
meth_df.columns = [extract_gene(col) for col in meth_df.columns]

# controleer met de volgende regel:
# print("Voorbeeld kolomnamen na omzetting:", meth_df.columns[:5])


In [150]:
import pandas as pd 

# aanmaken variabele met bestandspad
gmt_file = "../docs/h.all.v2025.1.Hs.symbols.gmt.txt"

# inlezen
import sys
sys.path.append("../scripts")

from utils import read_gmt

pathways = read_gmt(gmt_file)

# controle
print("Aantal pathways ingelezen:", len(pathways))
print("Eerste 5 pathways:", list(pathways.keys())[:5])

first_pathway = list(pathways.keys())[0]
print(f"Aantal genen in {first_pathway}: {len(pathways[first_pathway])}")
print("Eerste 5 genen:", pathways[first_pathway][:5])


Aantal pathways ingelezen: 50
Eerste 5 pathways: ['HALLMARK_ADIPOGENESIS', 'HALLMARK_ALLOGRAFT_REJECTION', 'HALLMARK_ANDROGEN_RESPONSE', 'HALLMARK_ANGIOGENESIS', 'HALLMARK_APICAL_JUNCTION']
Aantal genen in HALLMARK_ADIPOGENESIS: 200
Eerste 5 genen: ['ABCA1', 'ABCB8', 'ACAA2', 'ACADL', 'ACADM']


Waarnemingen:
Na het inlezen van de Hallmark gene sets blijken alle 50 pathways aanwezig te zijn. Ter controle is de eerste pathway (HALLMARK_ADIPOGENESIS) geinspecteerd. Deze bevat 200 genen, waaronder bekende genen zoals *ABCA1, ABCB8, ACAA2, ACADL en ACADM*. Dit bevestigs dat de koppeling tussen MSigDB en de gen-namen in de dataset correct is verlopen.

#### Samenvattende statistiek + overlap van genen

Voordat de daadwerkelijke aggregatie wordt uitgevoerd, is het belangrijk om inzicht te krijgen in de samenstelling van de pathways. Daarom wordt er eerst gekeken naar de grootte van elke pathway (het aantal genen) en de mate waarin genen in meerdere pathways voorkomen. 

Deze analyse geeft inzicht in redundantie en overlap tussen pathways. Dit is relevant omdat overlappende genen kunnen leiden tot gecorreleerde pathway-features. Dit heeft invloed op de interpretatie van het model en de feature importance in latere analyses. ******

In [151]:
import pandas as pd
from collections import Counter


# keys = pathway names, values = lijst met genen
num_genes = [len(genes) for genes in pathways.values()]
df_pathways = pd.DataFrame({"Pathway": list(pathways.keys()), "NumGenes": num_genes})

# bereken samenvattende statistiek
n_pathways = len(df_pathways)
mean_genes = round(df_pathways["NumGenes"].mean())
std_genes = round(df_pathways["NumGenes"].std())
min_genes = int(df_pathways["NumGenes"].min())
max_genes = int(df_pathways["NumGenes"].max())

# berekenen informatie over overlap tussen pathways
all_genes = [gene for genes in pathways.values() for gene in genes]
gene_counts = Counter(all_genes)
shared_genes = sum(1 for g, c in gene_counts.items() if c > 1)
unique_genes = sum(1 for g, c in gene_counts.items() if c == 1)
total_genes = len(gene_counts)
perc_shared = round(shared_genes / total_genes * 100)
perc_unique = round(unique_genes / total_genes * 100)

# genereer output voor verslag (reproduceerbaar)
# 
if std_genes > 0:
    summary_text = (
        f"Het aantal genen per pathway varieert van {min_genes} tot {max_genes} genen, "
        f"met gemiddeld {mean_genes} genen per pathway (± {std_genes}). "
    )
else:
    summary_text = (
        f"Alle {n_pathways} pathways bevatten hetzelfde aantal genen ({mean_genes}). "
    )

overlap_text = (
    f"Van de {total_genes} unieke genen die in de {n_pathways} Hallmark pathways voorkomen, "
    f"komen er {shared_genes} genen ({perc_shared}%) in meerdere pathways voor, "
    f"terwijl {unique_genes} genen ({perc_unique}%) slechts in één pathway aanwezig zijn."
)
    

# print alles 
full_text = summary_text + overlap_text
print(full_text)

Het aantal genen per pathway varieert van 32 tot 200 genen, met gemiddeld 146 genen per pathway (± 62). Van de 4384 unieke genen die in de 50 Hallmark pathways voorkomen, komen er 1713 genen (39%) in meerdere pathways voor, terwijl 2671 genen (61%) slechts in één pathway aanwezig zijn.


Conclusie:
Bij de aggregatie van genen naar pathway-level features blijkt dat een groot deel van de genen in meerdere pathways voorkomt, terwijl andere genen slechts in één pathway aanwezig zijn. Deze overlap is biologisch verklaarbaar, aangezien genen vaak meerdere processen reguleren. Hierdoor zijn sommige pathway-scores deels gecorreleerd: verandering in één gen kunnen meerdere pathway-features tegelijk beïnvloeden. 

In tegenstelling tot andere ML-modellen kan Random Forest goed omgaan met deze gecorreleerde features. Random Forest combineert meerdere decision trees die elk splitsingen maken op individuele features. [dit nog uitgebreider uitleggen]

#### Aggregatie naar pathway level features


In deze stap worden individuele genen (en bij methylatie: CpG-loci die aan genen zijn gekoppend) samengevat tot pathway-level features. Voor elke Hallmark-pathway wordt één nieuwe feature aangemaakt die de status van dat biologische proces per cellijn representeert.

De oorspronkelijke mutatiedata bestaat uit binaire waarden per gen (0 = geen mutatie, 1 = mutatie aanwezig). Bij aggregatie op pathway-niveau worden deze binaire waarden samengevat door het gemiddelde te nemen over alle genen die tot een pathway behoren. Hierdoor ontstaan per cellijn een continue pathwayscore tussen 0 en 1, die aangeeft welk deel van de genen in het pathway gemuteerd is.

Voor metylatie geldt dat de oorspronkelijke waarden al continue zijn. Het gemiddelde per pathway vat hier de algemene methylatiestatus van de betrokken genen samen en geeft een globale indicatie van mogelijke pathway activatie of repressie.

In deze analyse is gekozen voor het gemiddelde als samenvattende maat omdat deze:  
- informatie over alle genen in het pathway behoudt,
- gevoeliger is voor sterke afwijkingen die biologisch relevant kunnen zijn,
- en het resulteert in continue features op een vergelijkbare schaal voor mutatie en methylatie. 

Alternatieve samenvattende maten, zoals mediaan of variantie per pathway, worden niet in de eerste analyse toegepast, maar zijn wel een mogelijkheid in een latere fase van het project.

Ter controle wordt de vorm van de resulterende pathway-matrix weergeven en een voorbeeld van de eerste twee mutatie-pathway features en de eerste twee methylatie-pathwayfeatures. Op basis van de Hallmark gene sets wordt verwacht dat de matrix 50 mutaties- en 50 methylatie-pathway features bevat (in totaal dus 100 featuers).


In [152]:
# daadwerkelijke aggregatie naar pathway-level features

# gebruik exact dezelfde cellijnen als in combined_df
mut_subset = mut_df.loc[common_cell_lines]
meth_subset = meth_df.loc[common_cell_lines]

# aanmaken lege dataframes voor pathway-features
mut_pathway_df = pd.DataFrame(index=mut_subset.index)
meth_pathway_df = pd.DataFrame(index=meth_subset.index)

# loop over pathways
for pathway, genes in pathways.items():

    # ---- Mutaties ----
    mut_genes = [g for g in genes if g in mut_subset.columns]
    if len(mut_genes) > 0:
        mut_pathway_df[f"MUT_{pathway}"] = mut_subset[mut_genes].mean(axis=1)

    # ---- Methylatie ----
    meth_genes = [g for g in genes if g in meth_subset.columns]
    if len(meth_genes) > 0:
        meth_pathway_df[f"METH_{pathway}"] = meth_subset[meth_genes].mean(axis=1)

# combineer mutatie- en methylatie-pathway-features
pathway_features_df = pd.concat([mut_pathway_df, meth_pathway_df], axis=1)

# verwijder 'HALLMARK_' uit kolomnamen voor leesbaarheid
pathway_features_df.columns = (
    pathway_features_df.columns
    .str.replace("HALLMARK_", "", regex=False)
)

# overzicht output (controle)
print("Vorm van pathway-level feature matrix:", pathway_features_df.shape)

# selecteer eerste 2 mutatie- en eerste 2 methylatie-pathway scores
mut_cols = [c for c in pathway_features_df.columns if c.startswith("MUT_")][:2]
meth_cols = [c for c in pathway_features_df.columns if c.startswith("METH_")][:2]

preview_cols = mut_cols + meth_cols

# toon eerste 5 cellijnen in tabelvorm
display(pathway_features_df[preview_cols].head())

Vorm van pathway-level feature matrix: (82, 100)


,MUT_ADIPOGENESIS,MUT_ALLOGRAFT_REJECTION,METH_ADIPOGENESIS,METH_ALLOGRAFT_REJECTION
ModelID,,,,
ACH-000012,0.015873,0.012048,0.272330,0.434992
ACH-000035,0.000000,0.000000,0.172558,0.392030
ACH-000062,0.015873,0.012048,0.166824,0.325776
ACH-000066,0.000000,0.012048,0.173513,0.336316
ACH-000161,0.015873,0.000000,0.285914,0.373604


De output laat zien dat de aggregatie naar pathway-niveau succesvol is uitgevoerd. De resulterende dataset bevat uitsluitend cellijnen die in alle drie de oorpronkelijke datasets aanwezig zijn. Door aggregatie is het aantal features sterk gereduceerd ten opzichte van de oorspronkelijke gen- en CpG-level data, terwijl de biologische interpretatie behouden blijft.

De voorbeeldweergave van de matrix bevestigt dat:  
- elke rij één longkankercellijn representeert (`ModelID`),
- de mutatie- en methylatie-features duidelijk van elkaar te onderscheiden zijn,
- en elke feature overeenkomt met één biologische pathway.

Hiermee is een compacte en overzichtelijke featurematrix verkregen die geschikt is als input voor machine learning.

De pathway-level featurematrix wordt opgeslagen als pickle-bestand (`pathway_features.pkl`) in de map `data/processed`, zodat deze direct herbruikbaar is voor het trainen en evalueren van machine learning-modellen.

De resulterende pathway-level featurematrix wordt opgeslagen als pickle-bestand (`pathway_features.pkl`) in de map `data/processed`. 

In [153]:
# opslaan van pathway-level feature matrix voor ML
pathway_features_df.to_pickle("../data/processed/pathway_features.pkl")

print("Pathway-level featurematrix opgeslagen als:")
print("data/processed/pathway_features.pkl")

Pathway-level featurematrix opgeslagen als:
data/processed/pathway_features.pkl


**Scheidinig tussen inputfeatures (X) en responswaarden (y)**

De responsdata (`lung_response_df.pkl`) bevat bewust nog meerdere geneesmiddelen per cellijn. Deze keuze is gemaakt om de pipeline reproduceerbaar en flexibel te houden: bij het trainen van een machine learning-model kan binnen scikit-learn eenvoudig één specifiek geneesmiddel worden geselecteerd als doelvariabele (y), zonder dat de onderliggende data opnieuw hoeft te worden geprepareerd. 

De afweging en selectie van het geneesmiddel is onderbouwd in het notebook `dataprep_response_df.ipynb`. Daar is vastgesteld dat slechts een beperkt aantal geneesmiddelen volledige responswaarden bevat voor alle longkankercellijnen. Op basis van datacompleetheid en biologisch werkingsmechanisme zijn twee kandidaten geselecteerd, waarbij eerst wordt gestart met PF-05212384. Een tweede geneesmiddel (Berzosertib) is benoemd als mogelijke vervolganalyse.

Door de responsdata gescheiden te houden van de featurematrix (X) en pas bij modeltrianing een specifieke keuze te maken voor `y`, blijft de pipeline overzichtelijk en herbruikbaar met de werkwijze van supervised machine learning in scikit-learn.

**Beantwoording van de deelvraag**

De deelvraag *“Hoe kan cellijn-data (mutaties, methylatie, drug-respons) worden verzameld, opgeschoond en geschikt gemaakt voor machine learning?”* is in dit notebook (met bijbehorende subnotebooks) is beantwoord door het opzetten van een volledige en reproduceerbare datapreprocessing- en integratiepipeline.

In afzonderlijke notebooks zijn:
- mutatie-, methylatie-- en responsdata ingelezen en verkend,
- longkankercellijnen geselecteerd,
- kwaliteitscontrole uitgevoerd,
- cellijnen geharmoniseerd met behulp van standaard DepMap-identifiers,
- en moleculaire data geaggregeerd naar pathway-level features.

In dit notebook zijn deze datasets vervolgens geintegreerd tot één consistente featurematrix die geschikt is als input voor supervised machine learning. De resulterende dataset vormt de basis voor de vervolganalyse, waarin een Random Forest model wordt getraind en geëvauleerd om verbanden tussen moleculaire kenmerken en therapierespons te onderzoeken.

